# cortrix-skills · Claude Tools demo

Use Cortrix as native tools in the Anthropic Messages API with `as_claude_tools(kit)`.

> **Skeleton notebook.** Cells are the canonical usage flow. A real run needs a live
> cortrix-server + an Anthropic API key (Claude tool-use round-trip) and is exercised
> during integration (integration), not in standalone development.

## Install

```bash
pip install cortrix-skills[claude]
```

In [ ]:
from cortrix_skills import CortrixToolKit
from cortrix_skills.adapters import as_claude_tools

kit = CortrixToolKit(
    base_url="https://cortrix.example.com",
    api_key="sk-cortrix-...",
)

tools = as_claude_tools(kit)
print(f"{len(tools)} Claude tool definitions")
tools[0]  # {name, description, input_schema}

In [ ]:
from anthropic import Anthropic
from cortrix_skills.adapters.claude import dispatch_claude_tool_use

client = Anthropic(api_key="sk-ant-...")
messages = [{"role": "user", "content": "find last week's notes on the MCP design"}]

resp = client.messages.create(model="claude-...", max_tokens=4096, tools=tools, messages=messages)

# Run each tool_use block and feed the results back to Claude.
tool_results = [
    dispatch_claude_tool_use(kit, block)
    for block in resp.content
    if block.type == "tool_use"
]
messages.append({"role": "assistant", "content": resp.content})
messages.append({"role": "user", "content": tool_results})

final = client.messages.create(model="claude-...", max_tokens=4096, tools=tools, messages=messages)
print(final.content)

## Errors

On a Cortrix error, `dispatch_claude_tool_use` returns a `tool_result` block with
`is_error=True` and the four GEN-Agent fields as JSON content.